# Lab 8 · Chuỗi thời gian: quý, cửa sổ trượt và so cùng kỳ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 8**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook Bài 8 làm việc theo **tháng**. Lab này chuyển sang hai độ phân giải khác:
**ngày** (cửa sổ trượt 7 ngày) và **quý**, sau đó tính phép so cùng kỳ.

**Mục tiêu bài lab**

1. Cắt lát thời gian bằng chuỗi (`.loc["2019"]`) và đối chiếu với con số đã biết.
2. Tổng hợp theo quý (`resample`); nhận diện và loại kỳ chưa trọn ở cấp quý.
3. Làm mượt chuỗi ngày bằng cửa sổ trượt (`rolling`) và đọc điểm cao/thấp.
4. Tính so với cùng kỳ năm trước và phân biệt với so kỳ liền trước.

Phiên bản đề: `time-series-v1` (25/09/2026). Lưu đúng tên **`lab-08.ipynb`**.

## Cách làm và nộp bài

- Phần khởi động (Bài 1–2) và các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở:
  được dùng AI, kèm trách nhiệm khai báo theo chính sách AI của môn.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với input riêng, không phụ thuộc biến `rv`, `r`, `quy`, `SNAPSHOT`… ở cell khác.
  `pd` và `np` được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên dữ liệu giả lập; test ẩn dùng chuỗi thời gian khác cùng hợp đồng
  (mốc chụp khác, năm có ngày trống, năm nhuận…).
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Không sửa DataFrame/Series đầu vào (không `inplace=True`); không ghi cứng kết quả hay nhãn ngày.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1, 2, 4, 5, 6 — mỗi bài 10 điểm | 50 | Tự động |
| Bài 3 (quý và kỳ chưa trọn), Bài 7 (so kỳ trước/cùng kỳ) — mỗi bài 15 điểm | 30 | Tự động |
| Diễn giải: kỳ chưa trọn, cực trị, cửa sổ trượt, hai phép so | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Yêu cầu dùng cắt lát chuỗi, `resample`, `rolling` được giảng viên kiểm tra cùng code.
Bài tự làm ✅ mở không cộng vào 80 điểm tự động.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định `USE_SNAPSHOT = False`: notebook tạo **dữ liệu giả lập** (~11 nghìn dòng `listing_id`, `date`,
01/01/2019–01/07/2026) theo một quy tắc cố định có mùa vụ, chu kỳ tuần, giai đoạn phong toả và một ngày kỷ lục.
Không dùng kết quả giả lập để kết luận về Santiago.

Đặt `USE_SNAPSHOT = True` để đọc `visualisations/reviews.csv` của snapshot Santiago **2026-06-29**
(690.112 dòng). Mốc chụp dữ liệu `SNAPSHOT` dùng chung cho cả hai nguồn. Public checks luôn dùng dữ liệu giả lập.

In [ ]:
import numpy as np
import pandas as pd

# Hành vi pandas 3 (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

SNAPSHOT = "2026-06-29"
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/reviews.csv")

# Dữ liệu GIẢ LẬP: số đánh giá mỗi ngày theo một quy tắc cố định, rồi nhân ra thành từng dòng.
def so_danh_gia_gia_lap(ngay):
    muc = {2019: 3, 2020: 2, 2021: 2, 2022: 3, 2023: 4, 2024: 4, 2025: 5, 2026: 8}[ngay.year]
    if ngay.year == 2020 and 3 <= ngay.month <= 6:
        muc = 0                                   # phong toả: gần như không có khách
    if ngay.month in (1, 2):
        muc += 2                                  # mùa hè Nam bán cầu
    if ngay.dayofweek == 0:
        muc += 2                                  # thứ Hai: khách trả phòng sau cuối tuần
    if pd.Timestamp("2025-11-20") <= ngay <= pd.Timestamp("2025-11-26"):
        muc += 6                                  # một tuần có sự kiện lớn
    if ngay == pd.Timestamp("2026-03-16"):
        muc += 40                                 # ngày kỷ lục (xem Bài tự làm)
    return muc

_ngay = pd.date_range("2019-01-01", "2026-07-01", freq="D")
_so = [so_danh_gia_gia_lap(ngay) for ngay in _ngay]
DEMO_REVIEWS = pd.DataFrame({"date": _ngay.repeat(_so)})
DEMO_REVIEWS.insert(0, "listing_id", [100 + k % 37 for k in range(len(DEMO_REVIEWS))])
# Trộn thứ tự dòng như file thật (không sắp theo ngày).
DEMO_REVIEWS = DEMO_REVIEWS.sample(frac=1, random_state=8).reset_index(drop=True)

# Đặt USE_SNAPSHOT = True để làm trên 690.112 đánh giá Santiago.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    rv_raw = pd.read_csv(URL, parse_dates=["date"])
    DATA_LABEL = "Snapshot Santiago 2026-06-29"
else:
    rv_raw = DEMO_REVIEWS.copy()
    DATA_LABEL = f"GIẢ LẬP — {len(DEMO_REVIEWS):,} dòng"

print(DATA_LABEL, "·", rv_raw["date"].min().date(), "→", rv_raw["date"].max().date())

## Bài 1 · Khởi động: loại ngày sau mốc chụp, đưa `date` vào chỉ mục

File review có vài dòng **sau** ngày chụp dữ liệu (lỗi múi giờ/ghi nhận muộn). Loại chúng trước, rồi đưa `date` vào
chỉ mục và sắp xếp để các phép cắt lát và `resample` hoạt động.

### Hợp đồng hàm · 10 điểm tự động

`prepare_time_index(rv: pd.DataFrame, snapshot: str) -> dict`

Nhận bảng có cột `date` (`datetime64`, chưa sắp xếp) cùng các cột khác, và mốc `"YYYY-MM-DD"`. Trả dict có đúng các khóa:

- `so_dong_sau_moc`: int, số dòng có `date > snapshot`;
- `r`: DataFrame chỉ gồm các dòng `date <= snapshot`, **chỉ mục là `date`** (`set_index("date")`, cột `date`
  không còn trong các cột), `sort_index()` tăng dần, giữ mọi cột khác.

Không sửa `rv` (không `inplace=True`). Đối chiếu snapshot: **204** dòng sau mốc chụp.

In [ ]:
def prepare_time_index(rv: pd.DataFrame, snapshot: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    before = DEMO_REVIEWS.copy()
    result = prepare_time_index(DEMO_REVIEWS, SNAPSHOT)
    assert set(result) == {"so_dong_sau_moc", "r"}
    assert result["so_dong_sau_moc"] == 16, "30/06 và 01/07/2026 nằm sau mốc 29/06"
    r = result["r"]
    assert isinstance(r.index, pd.DatetimeIndex) and r.index.is_monotonic_increasing
    assert list(r.columns) == ["listing_id"] and len(r) == len(DEMO_REVIEWS) - 16
    assert r.index.max() == pd.Timestamp(SNAPSHOT)
    pd.testing.assert_frame_equal(DEMO_REVIEWS, before, obj="DEMO_REVIEWS (không được sửa)")

public_check("Q1 · ví dụ công khai", check_q1)

In [ ]:
def thuc_hanh_q1():
    global r
    kq = prepare_time_index(rv_raw, SNAPSHOT)
    r = kq["r"]
    return f"Đã loại {kq['so_dong_sau_moc']} dòng sau mốc chụp; còn {len(r):,} dòng, {r.index.min().date()} → {r.index.max().date()}"

preview("Bài 1 trên dữ liệu đang dùng", thuc_hanh_q1)

## Bài 2 · Khởi động: cắt lát bằng chuỗi + đối chiếu số đã biết

Ở lab 2 bạn đã đếm bằng dict: 2019 có **24.299** đánh giá, 2021 có **23.926**. Đếm lại bằng cắt lát chuỗi trên `r`.

### Hợp đồng hàm · 10 điểm tự động

`count_year(r: pd.DataFrame, nam: int) -> int`

Nhận bảng có `DatetimeIndex` đã sắp xếp và một năm có ít nhất một dòng trong bảng.
Trả **int**: số dòng của năm đó, lấy bằng cắt lát chuỗi `r.loc[str(nam)]`. Không sửa input.

In [ ]:
def count_year(r: pd.DataFrame, nam: int) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    r_demo = DEMO_REVIEWS.set_index("date").sort_index()
    n_2019, n_2021 = count_year(r_demo, 2019), count_year(r_demo, 2021)
    assert isinstance(n_2019, (int, np.integer)) and not isinstance(n_2019, bool)
    assert n_2019 == 1317 and n_2021 == 952
    assert count_year(r_demo, 2020) == int((DEMO_REVIEWS["date"].dt.year == 2020).sum())

public_check("Q2 · ví dụ công khai", check_q2)

In [ ]:
def thuc_hanh_q2():
    n_2019, n_2021 = count_year(r, 2019), count_year(r, 2021)
    return f"2019: {n_2019:,} · 2021: {n_2021:,} — đến 2021 thị trường đã hồi {n_2021 / n_2019:.0%} mức 2019."

preview("Bài 2 trên dữ liệu đang dùng", thuc_hanh_q2)

## Bài 3 · Tổng hợp theo quý và nhận diện kỳ chưa trọn

`resample("QE")` gán nhãn mỗi quý bằng **ngày cuối quý** (ví dụ Q2/2026 → `2026-06-30`). Nếu mốc chụp dữ liệu
đứng trước ngày cuối quý, quý đó **chưa trọn** — dù số lượng trông có vẻ gần đủ.

### Hợp đồng hàm · 15 điểm tự động

`quarterly(r: pd.DataFrame, snapshot: str) -> dict`

Nhận bảng có `DatetimeIndex` đã sắp xếp (mọi ngày `<= snapshot`) và mốc chụp. Trả dict có đúng các khóa:

- `quy`: Series `r.resample("QE").size()` — số dòng mỗi quý, gồm cả quý không có dòng nào (0) ở giữa;
- `quy_day_du`: các phần tử của `quy` có nhãn (ngày cuối quý) `<= snapshot`, tức bỏ quý chưa trọn cuối cùng
  nếu có; nếu `snapshot` đúng là ngày cuối quý thì giữ nguyên.

Không ghi cứng nhãn quý. Đối chiếu snapshot: Q1/2026 (`2026-03-31`) có **68.395**; Q2/2026 chưa trọn có **60.064**.

In [ ]:
def quarterly(r: pd.DataFrame, snapshot: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    r_demo = DEMO_REVIEWS.loc[DEMO_REVIEWS["date"] <= SNAPSHOT].set_index("date").sort_index()
    result = quarterly(r_demo, SNAPSHOT)
    assert set(result) == {"quy", "quy_day_du"}
    quy, quy_day_du = result["quy"], result["quy_day_du"]
    assert isinstance(quy, pd.Series) and len(quy) == 30
    assert quy.loc["2026-03-31"] == 904 and quy.loc["2026-06-30"] == 746
    assert len(quy_day_du) == 29 and quy_day_du.index[-1] == pd.Timestamp("2026-03-31")
    tron = quarterly(r_demo.loc[:"2025-12-31"], "2025-12-31")
    assert len(tron["quy_day_du"]) == len(tron["quy"]), "Mốc đúng ngày cuối quý: quý đó đã trọn"

public_check("Q3 · ví dụ công khai", check_q3)

**Kỳ chưa trọn:** quý cuối của dữ liệu thiếu những ngày nào? Nếu vẽ cả quý đó lên biểu đồ, người đọc sẽ hiểu sai thế nào? …

## Bài 4 · Quý cao nhất và quý thấp nhất một giai đoạn

### Hợp đồng hàm · 10 điểm tự động

`quarter_extremes(quy: pd.Series, bat_dau: str, ket_thuc: str) -> dict`

Nhận Series số đếm theo quý (index `DatetimeIndex` là ngày cuối quý, chỉ gồm quý trọn) và hai mốc chuỗi
(ví dụ `"2019"`, `"2021"`). Trả dict có đúng các khóa:

- `quy_dinh`: `Timestamp` nhãn của quý cao nhất **toàn chuỗi** (`idxmax`);
- `quy_day`: `Timestamp` nhãn của quý **thấp nhất trong giai đoạn** `quy.loc[bat_dau:ket_thuc]` (cắt lát trước, `idxmin` sau);
- `gia_tri_day`: int, số đếm của quý `quy_day`.

Đồng hạng lấy quý xuất hiện trước. Đối chiếu snapshot: cao nhất Q1/2026; thấp nhất 2019–2021 là Q2/2020 với **1.022**.

In [ ]:
def quarter_extremes(quy: pd.Series, bat_dau: str, ket_thuc: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    r_demo = DEMO_REVIEWS.set_index("date").sort_index()
    quy_demo = r_demo.loc[:"2026-03-31"].resample("QE").size()
    result = quarter_extremes(quy_demo, "2019", "2021")
    assert set(result) == {"quy_dinh", "quy_day", "gia_tri_day"}
    assert pd.Timestamp(result["quy_dinh"]) == pd.Timestamp("2026-03-31")
    assert pd.Timestamp(result["quy_day"]) == pd.Timestamp("2020-06-30") and result["gia_tri_day"] == 26
    toan_bo = quarter_extremes(quy_demo, "2019", "2026")
    assert pd.Timestamp(toan_bo["quy_day"]) == pd.Timestamp("2020-06-30")

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
def thuc_hanh_q3_q4():
    global quy
    kq = quarterly(r, SNAPSHOT)
    quy = kq["quy"]
    cuc_tri = quarter_extremes(kq["quy_day_du"], "2019", "2021")
    return (f"{quy.tail(4)}\n"
            f"Cao nhất: {cuc_tri['quy_dinh']:%Y-%m-%d} ({kq['quy_day_du'].max():,}) — thấp nhất 2019–2021: "
            f"{cuc_tri['quy_day']:%Y-%m-%d} ({cuc_tri['gia_tri_day']:,})")

preview("Bài 3–4 trên dữ liệu đang dùng", thuc_hanh_q3_q4)

## Bài 5 · Đếm theo ngày của một năm

Cắt lát năm **trước**, tổng hợp bằng `resample("D")` **sau** — thứ tự này tránh tạo hàng nghìn ngày ngoài năm cần xét.

### Hợp đồng hàm · 10 điểm tự động

`daily_counts(r: pd.DataFrame, nam: int) -> pd.Series`

Nhận bảng có `DatetimeIndex` đã sắp xếp và một năm có dữ liệu. Trả Series `r.loc[str(nam)].resample("D").size()`:
một phần tử cho mỗi ngày từ ngày **đầu tiên đến ngày cuối cùng có dữ liệu** của năm đó, ngày không có dòng nào là 0,
index là ngày. Không sửa input. Đối chiếu snapshot: năm 2025 có **365** ngày, ngày nhiều nhất **1.460** đánh giá.

In [ ]:
def daily_counts(r: pd.DataFrame, nam: int) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    r_demo = DEMO_REVIEWS.set_index("date").sort_index()
    ngay = daily_counts(r_demo, 2025)
    assert isinstance(ngay, pd.Series) and isinstance(ngay.index, pd.DatetimeIndex)
    assert len(ngay) == 365 and ngay.sum() == count_2025
    assert ngay.max() == 13 and ngay.loc["2025-11-24"] == 13
    assert (daily_counts(r_demo, 2020).loc["2020-04-01":"2020-04-05"] == 0).all(), "Ngày trống phải là 0"

count_2025 = int((DEMO_REVIEWS["date"].dt.year == 2025).sum())
public_check("Q5 · ví dụ công khai", check_q5)

## Bài 6 · Làm mượt bằng cửa sổ trượt

### Hợp đồng hàm · 10 điểm tự động

`rolling_peak(ngay: pd.Series, cua_so: int = 7) -> dict`

Nhận Series số đếm theo ngày (index ngày liên tiếp, ít nhất `cua_so` phần tử) và độ rộng cửa sổ.
Trả dict có đúng các khóa:

- `tb`: Series `ngay.rolling(cua_so, center=True).mean()` — cùng index, các ngày ở hai đầu không đủ cửa sổ là NaN;
- `ngay_dinh`: `Timestamp` ngày có `tb` lớn nhất (`idxmax`, bỏ NaN); `gia_tri_dinh`: float, giá trị `tb` tại đó.

Không sửa input. Đối chiếu snapshot: đỉnh năm 2025 theo trung bình 7 ngày là **23/11/2025**.

In [ ]:
def rolling_peak(ngay: pd.Series, cua_so: int = 7) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    r_demo = DEMO_REVIEWS.set_index("date").sort_index()
    ngay = r_demo.loc["2025"].resample("D").size()
    result = rolling_peak(ngay)
    assert set(result) == {"tb", "ngay_dinh", "gia_tri_dinh"}
    tb = result["tb"]
    assert isinstance(tb, pd.Series) and len(tb) == 365 and tb.isna().sum() == 6
    assert pd.Timestamp(result["ngay_dinh"]) == pd.Timestamp("2025-11-23")
    assert round(result["gia_tri_dinh"], 2) == 11.29
    assert pd.Timestamp(rolling_peak(ngay, 1)["ngay_dinh"]) == pd.Timestamp("2025-11-24"), "Cửa sổ 1 = không làm mượt"

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
def thuc_hanh_q5_q6():
    global ngay_2025, tb7
    ngay_2025 = daily_counts(r, 2025)
    kq = rolling_peak(ngay_2025)
    tb7 = kq["tb"]
    return (f"{len(ngay_2025)} ngày, nhiều nhất {ngay_2025.max():,} đánh giá/ngày · "
            f"đỉnh trung bình 7 ngày: {kq['ngay_dinh']:%d/%m/%Y} — {kq['gia_tri_dinh']:.0f} đánh giá/ngày")

preview("Bài 5–6 trên dữ liệu đang dùng", thuc_hanh_q5_q6)

In [ ]:
# Vẽ nhanh để quan sát cửa sổ trượt (mã cho sẵn — Bài 12 học kỹ hơn)
def ve_cua_so_truot():
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(ngay_2025.index, ngay_2025.values, color="#bbb", lw=0.8, label="theo ngày")
    ax.plot(tb7.index, tb7.values, color="#1E93AB", lw=2, label="trượt 7 ngày")
    ax.legend(frameon=False)
    ax.set_title("Số đánh giá theo ngày năm 2025 và trung bình trượt 7 ngày")
    plt.show()
    return "Đã vẽ."

preview("Biểu đồ", ve_cua_so_truot)

**Đọc cửa sổ trượt:** đường theo ngày dao động theo chu kỳ nào? Vì sao chọn cửa sổ 7 ngày? Đỉnh theo ngày và đỉnh theo
trung bình 7 ngày có trùng nhau không, vì sao? …

## Bài 7 · So cùng kỳ — con số báo cáo được

Tháng 5/2026 của snapshot có **22.296** đánh giá. Mức này cần được đặt cạnh một mốc so sánh phù hợp:
tháng liền trước, hay cùng tháng năm trước?

### Hợp đồng hàm · 15 điểm tự động

`period_changes(thang: pd.Series, moc: str) -> dict`

Nhận Series số đếm theo tháng (`r.resample("ME").size()`: index là **ngày cuối tháng**, liên tục) và nhãn tháng cần báo cáo
`moc` dạng ngày cuối tháng, ví dụ `"2026-05-31"`; chuỗi có đủ 12 tháng trước `moc`. Trả dict có đúng các khóa
(tính theo **%**, không làm tròn):

- `so_ky_truoc`: % thay đổi so với tháng liền trước (nhãn `pd.Timestamp(moc) - pd.offsets.MonthEnd(1)`);
- `so_cung_ky`: % thay đổi so với cùng tháng năm trước (nhãn `pd.Timestamp(moc) - pd.offsets.MonthEnd(12)`).

% thay đổi = `(giá trị kỳ này / giá trị mốc so − 1) × 100`. Có thể dùng `shift`/`pct_change` nếu kết quả đúng.
Đối chiếu snapshot: tháng 5/2026 là **+3,0%** so với tháng 4/2026 và **+53,7%** so với tháng 5/2025.

In [ ]:
def period_changes(thang: pd.Series, moc: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q7():
    r_demo = DEMO_REVIEWS.set_index("date").sort_index()
    thang_demo = r_demo.resample("ME").size()
    result = period_changes(thang_demo, "2026-05-31")
    assert set(result) == {"so_ky_truoc", "so_cung_ky"}
    assert round(result["so_ky_truoc"], 1) == 3.2 and round(result["so_cung_ky"], 1) == 57.1
    thang_2 = period_changes(thang_demo, "2021-02-28")
    assert round(thang_2["so_cung_ky"], 1) == round((thang_demo["2021-02-28"] / thang_demo["2020-02-29"] - 1) * 100, 1)

public_check("Q7 · ví dụ công khai", check_q7)

In [ ]:
def thuc_hanh_q7():
    thang = r.resample("ME").size()
    kq = period_changes(thang, "2026-05-31")
    return (f"T5/2026: {thang.loc['2026-05-31']:,} đánh giá · {kq['so_ky_truoc']:+.1f}% so tháng trước · "
            f"{kq['so_cung_ky']:+.1f}% so cùng kỳ")

preview("Bài 7 trên dữ liệu đang dùng", thuc_hanh_q7)

**Hai phép so:** mỗi con số trả lời câu hỏi gì? Với dữ liệu có mùa vụ, nên báo cáo con số nào để đánh giá thay đổi dài hạn? …

## Bài tự làm ✅ mở — không cộng vào 80 điểm tự động

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại yêu cầu đã gửi cho AI và cách kiểm chứng.

### Tự làm 1 · Điều tra ngày kỷ lục

Tìm **ngày có nhiều đánh giá nhất toàn bộ dữ liệu** (tổng hợp theo ngày bằng `resample("D")`). Trên snapshot, bạn sẽ thấy
một ngày giữa tháng 3/2026 với hơn 2.000 đánh giá, cao hơn rõ rệt các ngày quanh nó và là **thứ Hai**. Hãy:
(1) in số đánh giá của 5 ngày quanh nó; (2) tra thử tuần đó ở Santiago có sự kiện gì (gợi ý: một lễ hội âm nhạc lớn
thường diễn ra giữa tháng 3); (3) viết 2 câu kết luận kiểu báo cáo — có dè chừng đúng mực (đây là *một* cách giải thích,
chưa kiểm chứng độc lập).

### Tự làm 2 · Thời gian từ đánh giá gần nhất

Với bảng `listings` (bản rút gọn), trước hết loại `last_review` sau mốc chụp dữ liệu rồi tính số ngày
từ `last_review` đến 29/06/2026. Bao nhiêu chỗ ở có đánh giá gần nhất **cách đây hơn một năm**
(>365 ngày)? Đề xuất một cờ dữ liệu cho nhóm lâu không có đánh giá mới.

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu nhãn ngày, số đếm hoặc phép tính tay đã dùng để đối chiếu.

---

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Cắt lát chuỗi + đối chiếu số lab 2 | kiểm chứng chéo giữa các công cụ |
| Loại ngày sau mốc chụp dữ liệu và quý chưa trọn | mọi biểu đồ thời gian của bài tập lớn |
| Cửa sổ trượt 7 ngày (`rolling`) làm giảm chu kỳ tuần | trực quan hoá chuỗi thời gian (Bài 12) |
| So kỳ trước và so cùng kỳ | chọn mốc so sánh phù hợp |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Tuần tiếp theo là **thi giữa kỳ**. Sau giữa kỳ, Bài 10 tiếp tục với làm sạch dữ liệu có cấu trúc.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-08.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_08.ipynb`).
2. Hoàn thành hàm TODO, các diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-08/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-08` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()